# Temat 01 - praca w grupach

## Audyt nieznanego zbioru danych

**Czas:** 40-45 minut · **Grupy:** 3-4 osoby · **Efekt:** notatnik + karta zbioru danych

---

### Sytuacja

Jesteście zespołem analityków. Dostajecie plik z danymi i jedno zdanie od zleceniodawcy:
*„Zróbcie z tego model, potrzebujemy tego na wczoraj."*

Waszym zadaniem **nie jest** zrobienie modelu. Waszym zadaniem jest napisanie odpowiedzi,
która wyjaśni zleceniodawcy, co z tymi danymi jest nie tak i czego brakuje, żeby w ogóle
zacząć.

### Przydział zbiorów

| Grupa | Zbiór | Kolumna docelowa |
|---|---|---|
| 1 | `titanic` | `survived` |
| 2 | `mpg` | `mpg` |
| 3 | `planets` | `mass` |
| 4 | `taxis` | `fare` |
| 5 | `penguins` | `species` |
| 6 | `diamonds` | `price` |

*(jeżeli grup jest mniej, prowadząca wskazuje zbiory)*

---

### Zadania

**Zadanie 1 (10 min) - pierwszy kontakt**
Wczytajcie zbiór. Podajcie wymiary, typy kolumn i zużycie pamięci. Czy któraś kolumna
ma typ niezgodny z tym, co reprezentuje?

**Zadanie 2 (10 min) - braki**
Policzcie braki jawne i sprawdźcie, czy są braki ukryte pod tekstem. Która kolumna jest
najbardziej dziurawa? Czy da się ją uratować, czy trzeba ją usunąć?

**Zadanie 3 (10 min) - struktura**
Użyjcie funkcji `profil` z zajęć. Wskażcie:
- kolumny wyglądające na identyfikator,
- kolumny stałe,
- kolumny zduplikowane treściowo (dwie kolumny mówiące to samo),
- kolumny, które **nie mogą** trafić do modelu, bo zdradzają odpowiedź.

**Zadanie 4 (10 min) - reguły**
Zapiszcie **minimum cztery** reguły poprawności specyficzne dla waszego zbioru
i policzcie naruszenia.

**Zadanie 5 (5 min) - karta zbioru**
Wypełnijcie kartę zbioru danych. W sekcji „Znane ograniczenia" wpiszcie **trzy**
konkretne zastrzeżenia, które przekazalibyście zleceniodawcy.

---

### Prezentacja (2-3 minuty na grupę)

Każda grupa odpowiada na jedno pytanie: **„Czego nie wolno zrobić z tym zbiorem i dlaczego?"**

---

### Kryteria oceny

| Element | Punkty |
|---|---|
| Kod działa i jest czytelny | 2 |
| Wykryto braki jawne i ukryte | 2 |
| Poprawnie wskazano kolumny problematyczne (identyfikatory, duplikaty treści, wyciek) | 3 |
| Minimum 4 sensowne reguły dziedzinowe | 2 |
| Karta zbioru z trzema konkretnymi ograniczeniami | 2 |
| Jasna prezentacja wniosku | 1 |
| **Razem** | **12** |

In [1]:
# --- konfiguracja startowa (uruchom jako pierwszą komórkę) -------------------
# Działa w Google Colab i lokalnie. W Colab podłącza Dysk Google, odnajduje
# katalog kursu i ustawia go jako miejsce pracy - dzięki temu pliki zapisywane
# do katalogu 'wyniki' nie znikają po zamknięciu notatnika.
import sys
import os
from pathlib import Path

NAZWA_KURSU = "kurs_feature_engineering"

try:
    import google.colab  # noqa: F401
    W_COLAB = True
except ImportError:
    W_COLAB = False

if W_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")          # za pierwszym razem Colab poprosi o zgodę
    MOJ_DYSK = Path("/content/drive/MyDrive")

    # katalog kursu: w "Mój dysk" albo o jeden poziom głębiej
    kandydaci = [MOJ_DYSK / NAZWA_KURSU] + sorted(MOJ_DYSK.glob(f"*/{NAZWA_KURSU}"))
    KORZEN = next((k for k in kandydaci if (k / "dane.py").exists()), None)

    if KORZEN is None:
        # pierwsze uruchomienie: rozpakowujemy paczkę wgraną na Dysk
        import zipfile
        paczki = sorted(MOJ_DYSK.glob(f"{NAZWA_KURSU}*.zip"))
        if not paczki:
            raise FileNotFoundError(
                f"Nie znalazłam katalogu ani pliku {NAZWA_KURSU}.zip w 'Mój dysk'. "
                "Wgraj paczkę kursu do głównego katalogu Dysku Google i uruchom komórkę ponownie."
            )
        with zipfile.ZipFile(paczki[0]) as z:
            z.extractall(MOJ_DYSK)
        KORZEN = MOJ_DYSK / NAZWA_KURSU
        print("Rozpakowano paczkę kursu do:", KORZEN)

    os.chdir(KORZEN / "notatniki")
    sys.path.insert(0, str(KORZEN))
else:
    # praca lokalna: moduł dane.py leży w katalogu głównym materiałów
    for kandydat in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (kandydat / "dane.py").exists():
            sys.path.insert(0, str(kandydat))
            break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dane import wczytaj, lista_zbiorow

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 4.5)

RNG = 2026          # jedno ziarno losowości na cały notatnik - dla powtarzalności
np.random.seed(RNG)

print("Gotowe.", "Colab" if W_COLAB else "lokalnie", "| pandas", pd.__version__, "| numpy", np.__version__)


Mounted at /content/drive
Gotowe. Colab | pandas 2.2.3 | numpy 2.1.3


---
## Narzędzia z zajęć - gotowe do użycia

Poniższa komórka zawiera funkcje napisane wspólnie na zajęciach. Uruchom ją i korzystaj.

In [2]:
MARKERY_BRAKU = {"brak", "brak danych", "n/a", "n/d", "-", "?", "", "nan", "null", "bd", "x"}


def policz_braki_ukryte(ramka):
    wynik = {}
    for kol in ramka.columns:
        seria = ramka[kol].dropna().astype(str).str.strip().str.lower()   # dropna: braki jawne liczymy osobno
        wynik[kol] = seria.isin(MARKERY_BRAKU).sum()
    return pd.Series(wynik)


def profil(ramka, maks_przykladow=3):
    n = len(ramka)
    wiersze = []
    for kol in ramka.columns:
        s = ramka[kol]
        niepuste = s.notna().sum()
        rekord = {
            "kolumna": kol,
            "typ": str(s.dtype),
            "niepuste": niepuste,
            "braki": n - niepuste,
            "braki_%": round(100 * (n - niepuste) / n, 1),
            "unikalne": s.nunique(dropna=True),
            "unikalne_%": round(100 * s.nunique(dropna=True) / max(niepuste, 1), 1),
            "przyklady": ", ".join(map(str, s.dropna().unique()[:maks_przykladow])),
        }
        if pd.api.types.is_numeric_dtype(s) and niepuste > 0:
            rekord["min"] = round(float(s.min()), 2)
            rekord["mediana"] = round(float(s.median()), 2)
            rekord["max"] = round(float(s.max()), 2)
        wiersze.append(rekord)
    return pd.DataFrame(wiersze).set_index("kolumna")


def sprawdz_reguly(ramka, reguly):
    raport = []
    for opis, warunek in reguly.items():
        maska_ok = warunek(ramka)
        naruszenia = (~maska_ok & maska_ok.notna()).sum()
        raport.append({"regula": opis, "naruszenia": int(naruszenia),
                       "procent": round(100 * naruszenia / len(ramka), 2)})
    return pd.DataFrame(raport).sort_values("naruszenia", ascending=False)


print("Narzędzia gotowe: policz_braki_ukryte(), profil(), sprawdz_reguly()")
lista_zbiorow()[["zbior", "opis"]]

Narzędzia gotowe: policz_braki_ukryte(), profil(), sprawdz_reguly()


,zbior,opis
0,titanic,"Pasażerowie Titanica, 891 x 15. Braki w 'age' ..."
1,penguins,"Pingwiny z Palmer Station, 344 x 7. Czytelne b..."
2,diamonds,"Diamenty, 53 940 x 10. Skośne rozkłady, katego..."
3,tips,"Rachunki w restauracji, 244 x 7. Mały i czytel..."
4,taxis,"Przejazdy taksówek w Nowym Jorku, 6433 x 14. Z..."
5,mpg,"Spalanie samochodów, 398 x 9. Braki w 'horsepo..."
6,planets,"Egzoplanety, 1035 x 6. Bardzo skośne rozkłady ..."
7,healthexp,"Wydatki na zdrowie w krajach OECD, 274 x 4. Da..."
8,breast_cancer,"Diagnostyka guzów piersi, 569 x 31. Same cechy..."
9,wine,"Analiza chemiczna win, 178 x 14. Różne rzędy w..."


---
## Zadanie 1 - pierwszy kontakt

Wpiszcie nazwę swojego zbioru i ruszajcie.

In [3]:
NAZWA_ZBIORU = "titanic"      # <- ZMIEŃ na zbiór przydzielony waszej grupie

df = wczytaj(NAZWA_ZBIORU)

print("Wymiary:", df.shape)
df.info(memory_usage="deep")

Wymiary: (891, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  889 non-null    object 
 13  alive        891 non-null    object 
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 354.8 KB


In [4]:
df.head(10)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
5,0,3,male,NaN,0,0,8.4583,Q,Third,man,True,NaN,Queenstown,no,True
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
7,0,3,male,2.0,3,1,21.0750,S,Third,child,False,NaN,Southampton,no,False
8,1,3,female,27.0,0,2,11.1333,S,Third,woman,False,NaN,Southampton,yes,False
9,1,2,female,14.0,1,0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,False


## Zadanie 2 - braki jawne i ukryte

In [11]:
MARKERY_BRAKU = {"brak", "brak danych", "n/a", "n/d", "-", "?", "","nan", "null", "bd", "x"}


# FUNKCJA LICZĄCA BRAKI UKRYTE
def policz_braki_ukryte(ramka):
    wynik = {}
    for kol in ramka.columns:
        seria = (
            ramka[kol]
            .dropna()              # pomiń jawne NaN
            .astype(str)
            .str.strip()
            .str.lower()
        )
        wynik[kol] = seria.isin(MARKERY_BRAKU).sum()
    return pd.Series(wynik)

# BRKI JAWNE I UKRYTE
braki_jawne = df.isna().sum()
braki_ukryte = policz_braki_ukryte(df)

porownanie = pd.DataFrame({
    "jawne_NaN": braki_jawne,
    "ukryte_tekstem": braki_ukryte
})

porownanie["razem"] = porownanie.sum(axis=1)
porownanie["procent"] = (100 * porownanie["razem"] / len(df)).round(1)

print("\nBraki jawne i ukryte:")
print(porownanie[porownanie["razem"] > 0].sort_values("razem", ascending=False))


Braki jawne i ukryte:
             jawne_NaN  ukryte_tekstem  razem  procent
deck               688               0    688     77.2
age                177               0    177     19.9
embarked             2               0      2      0.2
embark_town          2               0      2      0.2


## Zadanie 3 - struktura zbioru

In [14]:
# Wasz kod
def profil(ramka, maks_przykladow=3):
    n = len(ramka)
    wiersze = []
    for kol in ramka.columns:
        s = ramka[kol]

        niepuste = s.notna().sum()

        rekord = {
            "kolumna": kol,
            "typ": str(s.dtype),
            "niepuste": niepuste,
            "braki": s.isna().sum(),
            "braki_%": round(s.isna().sum() / n * 100, 2),
            "unikalne": s.nunique(),
            "unikalne_%": round(s.nunique() / niepuste * 100, 2) if niepuste > 0 else 0,
            "przyklady": ", ".join(
                map(str, s.dropna().unique()[:maks_przykladow])
            ),
        }

        if pd.api.types.is_numeric_dtype(s) and niepuste > 0:
            rekord["min"] = round(float(s.min()), 2)
            rekord["mediana"] = round(float(s.median()), 2)
            rekord["max"] = round(float(s.max()), 2)
        wiersze.append(rekord)
    return pd.DataFrame(wiersze).set_index("kolumna")


profil(df)

,typ,niepuste,braki,braki_%,unikalne,unikalne_%,przyklady,min,mediana,max
kolumna,,,,,,,,,,
survived,int64,891,0,0.00,2,0.22,"0, 1",0.00,0.00,1.00
pclass,int64,891,0,0.00,3,0.34,"3, 1, 2",1.00,3.00,3.00
sex,object,891,0,0.00,2,0.22,"male, female",NaN,NaN,NaN
age,float64,714,177,19.87,88,12.32,"22.0, 38.0, 26.0",0.42,28.00,80.00
sibsp,int64,891,0,0.00,7,0.79,"1, 0, 3",0.00,0.00,8.00
parch,int64,891,0,0.00,7,0.79,"0, 1, 2",0.00,0.00,6.00
fare,float64,891,0,0.00,248,27.83,"7.25, 71.2833, 7.925",0.00,14.45,512.33
embarked,object,889,2,0.22,3,0.34,"S, C, Q",NaN,NaN,NaN
class,object,891,0,0.00,3,0.34,"Third, First, Second",NaN,NaN,NaN


**Wnioski zespołu (uzupełnijcie):**

- Kolumny wyglądające na identyfikator:
- Kolumny stałe:
- Kolumny mówiące to samo:
- Kolumny grożące wyciekiem informacji:

## Zadanie 4 - reguły poprawności

In [12]:
REGULY = {
    "wiek nieujemny":                 lambda d: d["age"] >= 0,
    "cena za bilet":                 lambda d: d["fare"] >= 0,
    "spójność zmiennych sibsp, parch i alone": lambda d: ((d["sibsp"] + d["parch"] > 0) & (d["alone"] == False)) | ((d["sibsp"] + d["parch"] ==0) & (d["alone"] == True)),
    "miasto wejścia na pokład":         lambda d: d["embark_town"].isin(["Cherbourg","Southampton","Queenstown"])

}

def sprawdz_reguly(ramka, reguly):
    raport = []
    for opis, warunek in reguly.items():
        maska_ok = warunek(ramka)
        # policz wiersze, które regułę łamią (pomiń NaN!)
        naruszenia = (maska_ok == False).sum()
        raport.append({
            "regula": opis,
            "naruszenia": int(naruszenia),
            "procent": round(100 * naruszenia / len(ramka), 2),
        })
    return pd.DataFrame(raport).sort_values("naruszenia", ascending=False)

sprawdz_reguly(df, REGULY)
    # "opis reguły": lambda d: warunek_zwracający_True_dla_poprawnych,


# sprawdz_reguly(df, REGULY)

,regula,naruszenia,procent
0,wiek nieujemny,177,19.87
3,miasto wejścia na pokład,2,0.22
1,cena za bilet,0,0.00
2,"spójność zmiennych sibsp, parch i alone",0,0.00


## Zadanie 5 - karta zbioru danych

Uzupełnijcie poniższą kartę. To jest wasz produkt końcowy - to ona idzie do zleceniodawcy.

In [13]:
from datetime import date

def karta_zbioru(ramka, nazwa, zrodlo, opis, uwagi_prawne=""):
    """Generuje szkielet karty zbioru danych w formacie Markdown."""
    p = profil(ramka)
    liczbowe = [k for k in ramka.columns if pd.api.types.is_numeric_dtype(ramka[k])]
    tekstowe = [k for k in ramka.columns if k not in liczbowe]
    return f"""
# Karta zbioru danych: {nazwa}

| Pole | Wartość |
|---|---|
| Data sporządzenia | {date.today().isoformat()} |
| Źródło | {zrodlo} |
| Liczba wierszy | {len(ramka)} |
| Liczba kolumn | {ramka.shape[1]} |
| Kolumny liczbowe | {len(liczbowe)} |
| Kolumny nieliczbowe | {len(tekstowe)} |
| Łączny odsetek braków | {round(100 * ramka.isna().sum().sum() / ramka.size, 2)}% |
| Kolumny z brakami | {', '.join(p[p['braki'] > 0].index) or 'brak'} |
| Duplikaty pełne | {int(ramka.duplicated().sum())} |

## Opis
{opis}

## Uwagi prawne i etyczne
{uwagi_prawne or '_do uzupełnienia_'}

## Znane ograniczenia
Zbiór zawiera braki danych, szczególnie w zmiennych Cabin (77,1%) i Age (19,9%). Niewielka liczba obserwacji (891 pasażerów) ogranicza możliwość uogólniania wyników.
Część zmiennych wymaga dodatkowego przetwarzania, a uzupełnianie brakujących wartości może wpływać na wyniki modeli.
Ponadto dane dotyczą jednego konkretnego wydarzenia historycznego, dlatego wykryte zależności nie muszą być reprezentatywne dla innych populacji.
"""

print(karta_zbioru(
    df,
    nazwa="titanic",
    zrodlo="Zbiór danych historycznych. Zawiera informacje, które opisują prawdziwych pasażerów podróżujących liniowcem RMS Titanic podczas jego tragicznego rejsu w 1912 roku.",
    opis= "Dane historyczne zawierające informacje o: przeżyciu, klasie pokładu, płci, wieku, liczbie małżonków lub rodzeństwa na pokładzie, liczbie rodziców lub dzieci na pokładzie, cene biletu oraz  port, w którym pasażer wsiadł na pokład",

    uwagi_prawne="Dane historyczne dotyczące pasażerów Titanica pochodzą z otwartych źródeł encyklopedycznych (np. Encyclopedia Titanica) oraz publicznych rejestrów brytyjskich i amerykańskich."
                  "Informacje te stanowią fakty historyczne, które nie podlegają ochronie prawa autorskiego."
                  "Zbiór nie spełnia kryteriów bazy danych chronionej prawem sui generis (na gruncie przepisów UE),"
                  "ponieważ jego kompilacja nie wymagała istotnego nakładu inwestycyjnego o charakterze komercyjnym od podmiotu trzeciego, a dane są powszechnie i darmowo dostępne."
))



# Karta zbioru danych: titanic
 
| Pole | Wartość |
|---|---|
| Data sporządzenia | 2026-09-26 |
| Źródło | Zbiór danych historycznych. Zawiera informacje, które opisują prawdziwych pasażerów podróżujących liniowcem RMS Titanic podczas jego tragicznego rejsu w 1912 roku. |
| Liczba wierszy | 891 |
| Liczba kolumn | 15 |
| Kolumny liczbowe | 8 |
| Kolumny nieliczbowe | 7 |
| Łączny odsetek braków | 6.5% |
| Kolumny z brakami | age, embarked, deck, embark_town |
| Duplikaty pełne | 107 |
 
## Opis
Dane historyczne zawierające informacje o: przeżyciu, klasie pokładu, płci, wieku, liczbie małżonków lub rodzeństwa na pokładzie, liczbie rodziców lub dzieci na pokładzie, cene biletu oraz  port, w którym pasażer wsiadł na pokład
 
## Uwagi prawne i etyczne
Dane historyczne dotyczące pasażerów Titanica pochodzą z otwartych źródeł encyklopedycznych (np. Encyclopedia Titanica) oraz publicznych rejestrów brytyjskich i amerykańskich.Informacje te stanowią fakty historyczne, które nie podlegają och

```
# Karta zbioru danych: ...................

| Pole | Wartość |
|---|---|
| Data sporządzenia |  |
| Źródło |  |
| Liczba wierszy |  |
| Liczba kolumn |  |
| Odsetek braków |  |
| Duplikaty pełne |  |

## Opis zbioru


## Uwagi prawne i etyczne


## Znane ograniczenia - trzy zastrzeżenia dla zleceniodawcy
1.
2.
3.

## Czego NIE WOLNO zrobić z tym zbiorem i dlaczego

```